# 03 - Baseline + GBDT Models

**Modelos en este notebook:**
1. Seed-only logistic regression (baseline floor)
2. LightGBM con todas las features
3. XGBoost
4. CatBoost
5. Feature importance analysis (SHAP)

**Evaluacion**: Temporal CV con Log Loss

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.calibration import calibration_curve
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

sys.path.insert(0, str(Path('.').resolve()))
from utils.cv import temporal_tournament_cv, evaluate_temporal_cv

# Add kaggle-pipeline to path for model wrappers
PIPELINE_DIR = Path('../../kaggle-pipeline')
sys.path.insert(0, str(PIPELINE_DIR.resolve()))
from src.models import LGBMWrapper, XGBWrapper, CatBoostWrapper
from src.ensemble import hill_climbing_weights, weighted_average

DATA_DIR = Path('../data')
RESULTS_DIR = Path('../results')
plt.style.use('seaborn-v0_8-whitegrid')
print('Imports OK')

In [ ]:
# Load training data from feature engineering notebook
train = pd.read_csv(DATA_DIR / 'train_matchups.csv')
print(f'Training data: {train.shape}')
print(f'Seasons: {train.Season.min()} - {train.Season.max()}')
print(f'Target: {train.target.value_counts().to_dict()}')

# Identify feature columns
meta_cols = ['Season', 'TeamID1', 'TeamID2', 'target']
feature_cols = [c for c in train.columns if c not in meta_cols]
print(f'\nFeatures: {len(feature_cols)}')
print(feature_cols)

In [ ]:
# Handle missing values
print('Missing values before fill:')
missing = train[feature_cols].isnull().sum()
print(missing[missing > 0])

# Fill NaN with 0 for difference features (means no information -> neutral)
train[feature_cols] = train[feature_cols].fillna(0)

# Prepare X, y, seasons
X = train[feature_cols].values
y = train['target'].values
seasons = train['Season']

print(f'\nX shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'y mean: {y.mean():.3f} (should be ~0.5)')

## 1. Baseline: Seed-only Logistic Regression

El seed difference solo, pasado por logistic regression, es un baseline muy fuerte.

In [ ]:
# Seed-only baseline
seed_col_idx = feature_cols.index('SeedNum_diff') if 'SeedNum_diff' in feature_cols else None

if seed_col_idx is not None:
    X_seed = X[:, [seed_col_idx]]
    
    def seed_model_fn(X_train, y_train):
        model = LogisticRegression(C=1.0, max_iter=1000)
        model.fit(X_train, y_train)
        return model
    
    print('=== Seed-only Logistic Regression ===')
    seed_results = evaluate_temporal_cv(
        seed_model_fn, X_seed, y, seasons, min_train_seasons=5
    )
    seed_oof = seed_results['oof_predictions']
else:
    print('SeedNum_diff not found in features!')

## 2. LightGBM

In [ ]:
# LightGBM with temporal CV
lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'learning_rate': 0.05,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'n_estimators': 500,
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1,
}

def lgb_model_fn(X_train, y_train):
    model = lgb.LGBMClassifier(**lgb_params)
    model.fit(X_train, y_train)
    return model

print('=== LightGBM ===')
lgb_results = evaluate_temporal_cv(
    lgb_model_fn, X, y, seasons, min_train_seasons=5
)
lgb_oof = lgb_results['oof_predictions']

## 3. XGBoost

In [ ]:
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 5,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 1.0,
    'n_estimators': 500,
    'tree_method': 'hist',
    'verbosity': 0,
    'random_state': 42,
    'n_jobs': -1,
}

def xgb_model_fn(X_train, y_train):
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X_train, y_train)
    return model

print('=== XGBoost ===')
xgb_results = evaluate_temporal_cv(
    xgb_model_fn, X, y, seasons, min_train_seasons=5
)
xgb_oof = xgb_results['oof_predictions']

## 4. CatBoost

In [ ]:
cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'Logloss',
    'learning_rate': 0.05,
    'depth': 6,
    'l2_leaf_reg': 3.0,
    'iterations': 500,
    'verbose': 0,
    'random_seed': 42,
}

def cb_model_fn(X_train, y_train):
    model = CatBoostClassifier(**cb_params)
    model.fit(X_train, y_train)
    return model

print('=== CatBoost ===')
cb_results = evaluate_temporal_cv(
    cb_model_fn, X, y, seasons, min_train_seasons=5
)
cb_oof = cb_results['oof_predictions']

## 5. Model Comparison

In [ ]:
# Compare all models
results_summary = {
    'Seed Baseline': seed_results['mean_score'] if seed_col_idx is not None else None,
    'LightGBM': lgb_results['mean_score'],
    'XGBoost': xgb_results['mean_score'],
    'CatBoost': cb_results['mean_score'],
}

print('\n=== MODEL COMPARISON ===')
print(f'{"Model":20s} | {"CV LogLoss":>12s}')
print('-' * 40)
for name, score in sorted(results_summary.items(), key=lambda x: x[1] if x[1] else 999):
    if score is not None:
        print(f'{name:20s} | {score:>12.5f}')

# Best model
best_model = min(results_summary, key=lambda k: results_summary[k] if results_summary[k] else 999)
print(f'\nBest model: {best_model} ({results_summary[best_model]:.5f})')

In [ ]:
# Calibration curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models_oof = {
    'LightGBM': lgb_oof,
    'XGBoost': xgb_oof,
    'CatBoost': cb_oof,
}

for idx, (name, oof) in enumerate(models_oof.items()):
    valid_mask = ~np.isnan(oof)
    prob_true, prob_pred = calibration_curve(y[valid_mask], oof[valid_mask], n_bins=10)
    axes[idx].plot(prob_pred, prob_true, 'o-', label=name)
    axes[idx].plot([0, 1], [0, 1], '--', color='gray')
    axes[idx].set_title(f'{name} Calibration')
    axes[idx].set_xlabel('Mean Predicted Probability')
    axes[idx].set_ylabel('Fraction of Positives')
    axes[idx].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (LightGBM)
# Train on all data to get importances
lgb_full = lgb.LGBMClassifier(**lgb_params)
lgb_full.fit(X, y)

importances = pd.DataFrame({
    'feature': feature_cols,
    'importance': lgb_full.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(12, 10))
importances.head(25).plot(kind='barh', x='feature', y='importance', ax=ax, color='steelblue')
ax.set_title('Top 25 Features (LightGBM Importance)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 25 features:')
print(importances.head(25).to_string(index=False))

In [ ]:
# Quick ensemble test
valid_mask = ~(np.isnan(lgb_oof) | np.isnan(xgb_oof) | np.isnan(cb_oof))

# Simple average
avg_oof = (lgb_oof[valid_mask] + xgb_oof[valid_mask] + cb_oof[valid_mask]) / 3
avg_score = log_loss(y[valid_mask], np.clip(avg_oof, 0.05, 0.95))
print(f'Simple average ensemble: {avg_score:.5f}')

# Hill climbing weights
best_weights = hill_climbing_weights(
    [lgb_oof[valid_mask], xgb_oof[valid_mask], cb_oof[valid_mask]],
    y[valid_mask],
    metric_fn=lambda y_t, y_p: log_loss(y_t, np.clip(y_p, 0.05, 0.95)),
    maximize=False,
    n_iter=200,
    step=0.02,
)

weighted_oof = weighted_average(
    [lgb_oof[valid_mask], xgb_oof[valid_mask], cb_oof[valid_mask]],
    best_weights
)
weighted_score = log_loss(y[valid_mask], np.clip(weighted_oof, 0.05, 0.95))

print(f'Hill climbing ensemble: {weighted_score:.5f}')
print(f'Weights: LGB={best_weights[0]:.3f}, XGB={best_weights[1]:.3f}, CB={best_weights[2]:.3f}')

In [ ]:
# Save results
results = {
    'models': results_summary,
    'ensemble_avg': avg_score,
    'ensemble_weighted': weighted_score,
    'weights': best_weights,
    'feature_cols': feature_cols,
    'top_features': importances.head(25)['feature'].tolist(),
}

with open(RESULTS_DIR / 'v1_baseline_results.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print('Results saved to results/v1_baseline_results.json')